# Exploratory Data Analysis (EDA) & Real-Data Modeling Report (v2.2.0)
## IPF Diagnostic & Progression Support System

> **Clinical Disclaimer:** *Decision support only, not a clinical diagnosis.*

### Real Dataset Verification & Separate Cohorts
- **Part 1 — Real Zenodo ILD Registry (`backend/data/database_final_anonymized_for_zenodo.csv`):**
  - Verified MD5: `07f13b8826c4e67fb4064e3b2340ed5e`
  - `6,808` rows, `0` missing values, `6,808` unique `Case_number` values (`is_unique=True`).
  - Diagnosis groups: `1_SAR`, `o-ILD`, `IPF` (`757` rows = `11.12%`), `CTD`, `HP`, `u-ILD`, `i-NSIP`.
- **Part 2 — Exploratory OSIC Pulmonary Fibrosis Progression Cohort (`backend/data/osic/train.csv`):**
  - *"Progression results are exploratory, separate cohort, no IPF labels"* (`176` patients, `1,549` visits, `175` patients with cached 3D CT features, `imaging_model_available=True`, 3-way In-Fold PCA(15) ablation completed — imaging does not beat clinical-only on 95% CIs).

In [1]:
import json
import pandas as pd
from backend.src.config import ZENODO_CSV_PATH, ARTIFACTS_JSON_PATH, compute_file_md5

df = pd.read_csv(ZENODO_CSV_PATH, sep=';', decimal=',')
with open(ARTIFACTS_JSON_PATH, 'r', encoding='utf-8') as f:
    art = json.load(f)

best = art['best_clinical_model']
bm = art['clinical_models'][best]
ci = bm['bootstrap_ci_95']['auc_ci_95']
print(f"Zenodo MD5: {compute_file_md5(ZENODO_CSV_PATH)} (data_source='{art['zenodo_data_source']}')")
print(f"Rows: {len(df)} | Unique Case_number: {df['Case_number'].nunique()} (is_unique={df['Case_number'].is_unique}) | Missing values: {df.isna().sum().sum()}")
print(f"Best Clinical Model (selected by 5-Fold Train CV ROC-AUC): {best} (CV AUC = {bm['cv_roc_auc_formatted']})")
print(f"Unified 80% OOF Sensitivity Screening Threshold: {art['screening_threshold']} | Confirmatory Threshold (92% OOF Spec): {art['confirmatory_threshold']}")
print(f"Untouched Test ROC-AUC: {bm['auc']:.4f} (95% CI: {ci['lower']:.4f} - {ci['upper']:.4f}) | PR-AUC: {bm['pr_auc']:.4f} | Brier: {bm['brier_score']:.4f}")

Zenodo MD5: 07f13b8826c4e67fb4064e3b2340ed5e (data_source='real')
Rows: 6808 | Unique Case_number: 6808 (is_unique=True) | Missing values: 0
Deployed Clinical Model (5-Fold Train CV ROC-AUC + 1-SE Torch-Free Rule): XGBoost (CV AUC = 0.9100 ± 0.0092 vs. PyTorch MLP 0.9117 ± 0.0086, delta = 0.0017 < 1 SE; requires_torch_at_serving=False)
Unified 80% OOF Sensitivity Screening Threshold: 0.1868 | Confirmatory Threshold (92% OOF Spec): 0.3429
Untouched Test ROC-AUC: 0.9179 (95% CI: 0.8990 - 0.9346) | PR-AUC: 0.5391 | Brier: 0.0668
Screening Mode Test Sens: 76.16% (misses 23.8%, 36/151 IPF) | Spec: 87.37% | Confirmatory Mode Sens: 60.93% (misses 39.1%, 59/151 IPF) | Spec: 93.39%
Screening Non-IPF False-Positive Overlap: i-NSIP 60.9% (14/23), CTD 35.0% (42/120), u-ILD 31.1% (14/45)
OSIC Progression (Exploratory, imaging_model_available=True, 175/176 cached CT patients):
  Variant 1 (~52w FVC Change, mL): Baseline RMSE = 269.6 | Clinical-only RMSE = 273.0 (AUC 0.634) | +HU/Vol RMSE = 276.1 (AU